# Cerulean API Guide

Explore **potential oil slick detections**, human reviews, and candidate source associations through [Cerulean's public OGC API](https://api.cerulean.skytruth.org/?utm_medium=notebook&utm_source=unidentified-colab).

Read the [Cerulean API guide](https://api.cerulean.skytruth.org/guide?utm_medium=notebook&utm_source=unidentified-colab) for queries and interpretation, and the [origin guide](https://api.cerulean.skytruth.org/guide/origin?utm_medium=notebook&utm_source=unidentified-colab) for origin estimates and source-type counts. Automated clients can also use the [investigation skill](https://api.cerulean.skytruth.org/skills/cerulean-investigation/SKILL.md?utm_medium=notebook&utm_source=unidentified-colab). The [source directory](https://github.com/SkyTruth/cerulean-cloud/tree/main/docs/agent-skills/cerulean-investigation) is also available for offline use. This notebook is maintained in the same repository as the API and skill.

For imagery, winds, tracks and source verification, use the [Cerulean map](https://cerulean.skytruth.org/?utm_medium=notebook&utm_source=unidentified-colab). SAR imagery alone cannot establish oil, volume, source responsibility, intent or legality. Counts below are **detection records, not distinct spills**.

## Interpretation rules

- Ignore the machine class `cls`. `machine_confidence` appears only in the map's legacy quality screen; do not interpret or rank by it.
- Use the latest `hitl_cls` / `hitl_cls_name` for human review. Say **human-classified apparent oil**, not confirmed oil. Not-oil detections should be excluded; `slick_plus` already excludes the current not-oil classes.
- `slick_plus.slick_confidence` is a geometric random-forest score, not an oil probability. Use it as a backup screen when source matching is absent. **Do not use `source_plus.slick_confidence`: it currently aliases the legacy machine score.**
- Collated scores rank candidate associations; they are not probabilities or proof of a discharge. Positive support for a possible origin means each candidate's own `source_collated_score > 0`.
- Missing AIS does not establish concealment. Multiple candidates of the same type do not make the origin mixed; multiple distinct positive-scoring types do.
- The API does not expose source verification/rejection fields. Check consequential associations on the map; API-only estimates may include reviewer-rejected associations.

In [ ]:
%pip -q install geopandas contextily

In [ ]:
import csv
import json
import shlex
import subprocess
import time
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box, shape
from shapely.ops import unary_union

API = "https://api.cerulean.skytruth.org/collections"
UTM_SOURCE = "unidentified-colab"  # Replace with your organization or public handle; no personal/confidential details.
AGENTIC_USE = False  # Agents running this notebook must set True and identify their organization/app.
UTM = {"utm_medium": "agentic" if AGENTIC_USE else "notebook", "utm_source": UTM_SOURCE}
OUTPUT_DIR = Path("cerulean-api-results")
OUTPUT_DIR.mkdir(exist_ok=True)

BBOX = "11,36,20,43"  # west,south,east,north; polygons intersect this box and are not clipped.
PERIOD = (
    "2025-05-01T00:00:00Z/2025-05-08T23:59:59Z"  # UTC acquisition times, inclusive.
)
SCREEN = "machine_confidence >= 0.7 AND max_source_collated_score >= 0"
SLICK_PROPERTIES = "id,slick_timestamp,s1_scene_id,area,hitl_cls,hitl_cls_name,slick_confidence,max_source_collated_score,aoi_type_1_ids,aoi_type_2_ids,aoi_type_3_ids,slick_url"
AREA_PARAMS = {
    "bbox": BBOX,
    "datetime": PERIOD,
    "filter": SCREEN,
    "properties": SLICK_PROPERTIES,
}
request_log = []
print("Attribution:", UTM)

## Reproducible requests and complete pagination

All Cerulean network requests below use `curl`. Python processes saved responses locally. Each attempt saves its raw response, exact command, HTTP status, UTC retrieval time and attribution in the results directory. Errors and incomplete downloads raise exceptions rather than becoming zero counts.

The API defaults to 10 rows and allows at most 50,000 per page. This helper requests 1,000 and follows offsets until the unique records received equal `numberMatched`. Slick IDs identify detections; `(slick_id, source_id)` identifies source associations. Keep raw pages separate from derived exports. The normal screen approximates the map, whose top-three and reviewer-verification rules can differ.

In [ ]:
def download(collection, params, name, suffix="geojson"):
    for attempt in range(3):
        output = (
            OUTPUT_DIR
            / f"{name}-offset-{params.get('offset', 0)}-attempt-{attempt}.{suffix}"
        )
        command = [
            "curl",
            "-sS",
            "--get",
            "--max-time",
            "90",
            f"{API}/{collection}/items",
        ]
        for key, value in {"f": "geojson", **params, **UTM}.items():
            command += ["--data-urlencode", f"{key}={value}"]
        command += ["-o", str(output), "-w", "%{http_code}"]
        completed = subprocess.run(command, check=True, capture_output=True, text=True)
        status = int(completed.stdout)
        request_log.append(
            {
                "command": shlex.join(command),
                "http_status": status,
                "retrieved_utc": datetime.now(timezone.utc).isoformat(),
                "raw_file": str(output),
                **UTM,
            }
        )
        (OUTPUT_DIR / "requests.json").write_text(json.dumps(request_log, indent=2))
        if status == 200:
            return output
        if status not in (429, 502, 503, 504) or attempt == 2:
            raise RuntimeError(
                f"HTTP {status}: inspect {output}; this is not an empty result"
            )
        time.sleep(2**attempt)


def fetch_features(collection, params, name, key_fields=("id",), sortby="id"):
    features, seen, offset, matched = [], set(), 0, None
    while True:
        page_path = download(
            collection,
            {**params, "limit": 1000, "offset": offset, "sortby": sortby},
            name,
        )
        page = json.loads(page_path.read_text())
        if matched is None:
            matched = page["numberMatched"]
        if page["numberMatched"] != matched:
            raise RuntimeError(
                "Population changed during pagination; retrieve a fresh snapshot"
            )
        rows = page["features"]
        for feature in rows:
            props = feature["properties"]
            key = tuple(
                (props.get("id", feature.get("id")) if field == "id" else props[field])
                for field in key_fields
            )
            if key not in seen:
                features.append(feature)
                seen.add(key)
        if len(seen) == matched:
            break
        if not rows or len(seen) > matched:
            raise RuntimeError(f"Incomplete {name}: received {len(seen)} of {matched}")
        offset += len(rows)
    print(f"{name}: {matched} matched; {len(seen)} unique received")
    return {
        "type": "FeatureCollection",
        "numberMatched": matched,
        "numberReturned": len(features),
        "features": features,
    }


def visualize(data):
    if not data["features"]:
        print(
            "No detections matched this complete query; this does not establish a clean area."
        )
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    frame = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
    ax = frame.to_crs(3857).plot(figsize=(10, 8), alpha=0.5, edgecolor="k")
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
    plt.show()
    return frame  # Keep downloaded geometries in geographic coordinates for exports.


def tagged_link(url):
    parts = urlsplit(url)
    if parts.scheme != "https" or parts.netloc not in (
        "api.cerulean.skytruth.org",
        "cerulean.skytruth.org",
    ):
        raise ValueError(
            "Only tag HTTPS links to Cerulean; do not send attribution to third parties"
        )
    query = [
        (key, value)
        for key, value in parse_qsl(parts.query, keep_blank_values=True)
        if key not in UTM
    ]
    return urlunsplit(parts._replace(query=urlencode(query + list(UTM.items()))))

## 1. Detections in an area and acquisition period

Bounding boxes use **west, south, east, north**. This bounded Mediterranean example applies the default API quality screen. Results intersect the box; they are not clipped. Never sum overlapping polygon areas as unique polluted area or interpret polygon area as oil volume.

In [ ]:
area_data = fetch_features("public.slick_plus", AREA_PARAMS, "area")
area_gdf = visualize(area_data)
print(
    "Detection IDs:", [feature["properties"]["id"] for feature in area_data["features"]]
)

### Coverage context

For small areas or periods shorter than a year, report the scene inventory's footprint coverage and acquisition gaps alongside detection results. Inventory footprints do not guarantee successful processing or usable imagery throughout each scene. Even 100% intersected once does not imply continuous monitoring. Overlapping slices from one satellite pass are not separate visits; the timestamps and gaps below describe inventory acquisitions.

In [ ]:
def coverage_context(bbox_value, period, name, aoi_outline=None):
    scenes = fetch_features(
        "public.sentinel1_grd",
        {
            "bbox": bbox_value,
            "datetime": period,
            "datetime-column": "start_time",
            "properties": "id,scene_id,start_time,end_time,absolute_orbit_number",
        },
        name,
    )
    bounds = tuple(map(float, bbox_value.split(",")))
    aoi_geometry = box(*bounds) if aoi_outline is None else shape(aoi_outline)
    aoi = gpd.GeoSeries([aoi_geometry], crs=4326).to_crs(6933).iloc[0]
    selected = [
        feature
        for feature in scenes["features"]
        if shape(feature["geometry"]).intersects(aoi_geometry)
    ]
    dates = sorted({feature["properties"]["start_time"] for feature in selected})
    fraction = 0.0
    if selected:
        footprints = gpd.GeoDataFrame.from_features(selected, crs=4326).to_crs(6933)
        fraction = (
            unary_union(list(footprints.geometry)).intersection(aoi).area / aoi.area
        )
    start, end = [
        datetime.fromisoformat(value.replace("Z", "+00:00"))
        for value in period.split("/")
    ]
    times = [
        datetime.fromisoformat(value.replace("Z", "+00:00")).replace(
            tzinfo=timezone.utc
        )
        for value in dates
    ]
    gaps = [
        (right - left).total_seconds() / 3600
        for left, right in zip([start] + times, times + [end])
    ]
    print("Cerulean inventory footprint fraction intersected:", round(fraction, 4))
    print("Acquisition times (UTC):", dates)
    print("Gaps including period boundaries (hours):", gaps)
    return scenes


area_scenes = coverage_context(BBOX, PERIOD, "area-coverage")

## 2. Prioritize candidate associations by collated score

Sorting by `max_source_collated_score` prioritizes candidate associations for review. It does not rank the probability of oil or confirm a source. The score is not a percentage, and no precision increase is guaranteed by raising a threshold.

In [ ]:
ranked_data = fetch_features(
    "public.slick_plus", AREA_PARAMS, "ranked", sortby="-max_source_collated_score,id"
)
pd.DataFrame([feature["properties"] for feature in ranked_data["features"]])

## 3. Add an area filter

`area` is polygon area in square meters. This deliberate extra filter selects polygons larger than 5 km²; report its count alongside the default count. Use URL-encoded CQL2 comparisons such as `>=`, `>`, `AND`, `IN`, and `IS NULL`. Array columns cannot be filtered by the API; filter them locally after a complete download. The API's `IS NOT NULL` behavior is unreliable, so avoid it.

In [ ]:
large_data = fetch_features(
    "public.slick_plus",
    {**AREA_PARAMS, "filter": SCREEN + " AND area > 5000000"},
    "large",
)
print(
    "Default:",
    area_data["numberMatched"],
    "Larger than 5 km²:",
    large_data["numberMatched"],
)

## 4. Recent detections: keep screened and provisional results separate

Use UTC acquisition timestamps. Source matching may arrive days later or be rerun much later; **72 hours is not a completion guarantee**, and an absent candidate does not establish whether AIS/dark analysis has run.

Retrieve the default screened population first. A second, explicitly provisional query admits detections with no collated score and a geometric score at least 0.8 (or absent), retaining the map's legacy machine threshold. The geometric score cannot establish a vessel origin. Keep the populations separate rather than using geometric scores to override poor source matches.

In [ ]:
now = datetime.now(timezone.utc).replace(microsecond=0)
recent_period = f"{(now - timedelta(hours=72)).isoformat().replace('+00:00', 'Z')}/{now.isoformat().replace('+00:00', 'Z')}"
recent_screened = fetch_features(
    "public.slick_plus", {**AREA_PARAMS, "datetime": recent_period}, "recent-screened"
)
recent_provisional = fetch_features(
    "public.slick_plus",
    {
        **AREA_PARAMS,
        "datetime": recent_period,
        "filter": "machine_confidence >= 0.7 AND max_source_collated_score IS NULL AND (slick_confidence >= 0.8 OR slick_confidence IS NULL)",
    },
    "recent-provisional",
)
recent_scenes = coverage_context(BBOX, recent_period, "recent-coverage")

## 5. Estimate possible origin from all positive-scoring source types

Human Vessel (including old/recent/coincident), Infrastructure or Natural review takes priority. Otherwise, collect **every distinct `source_type` whose own score is strictly greater than zero**, across all available ranks. One type supports its possible origin; several types give **Uncertain**. Several candidates of the same type still count as one type. Zero, negative and null scores do not qualify.

Keep `DARK` distinct from `VESSEL` for dark-vessel questions. A human Vessel classification does not establish DARK status. With infrastructure-only support before AIS/dark analysis is known to have run, use **Infrastructure (Pending)** only when an explicit pending signal exists; otherwise the API cannot determine processing status. No positive support gives **Pending** when pending is known, or **Unknown** when analysis has run/status is unavailable.

In [ ]:
def sources_for_slicks(slick_features, name):
    ids = [feature["properties"]["id"] for feature in slick_features]
    sources = []
    for batch_index, start in enumerate(range(0, len(ids), 100)):
        batch = ",".join(str(int(value)) for value in ids[start : start + 100])
        data = fetch_features(
            "public.source_plus",
            {
                "filter": f"slick_id IN ({batch}) AND source_collated_score > 0",
                "properties": "slick_id,source_id,source_type,mmsi_or_structure_id,source_rank,source_collated_score,create_time",
                "bbox-only": "true",
            },
            f"{name}-{batch_index}",
            key_fields=("slick_id", "source_id"),
            sortby="slick_id,source_id",
        )
        sources.extend(data["features"])
    return sources


def possible_origins(slick_features, source_features, analysis_status_by_slick=None):
    status_by_id = (
        analysis_status_by_slick or {}
    )  # Only explicit processing signals; never infer from age.
    types = defaultdict(set)
    for feature in source_features:
        row = feature["properties"]
        score = row["source_collated_score"]
        if score is not None and score > 0:
            types[row["slick_id"]].add(row["source_type"])
    human_origins = {
        3: "Natural",
        4: "Infrastructure",
        **dict.fromkeys((5, 6, 7, 8), "Vessel"),
    }
    labels = {
        "VESSEL": "Vessel",
        "INFRA": "Infrastructure",
        "DARK": "Dark",
        "NATURAL": "Natural",
    }
    results = []
    for feature in slick_features:
        slick = feature["properties"]
        review = slick["hitl_cls"]
        if review in (1, 10, 11, 12):
            continue
        source_types = types[slick["id"]]
        status = status_by_id.get(slick["id"], "unavailable")
        human = human_origins.get(review)
        if human:
            origin, basis = human, "human_review"
        elif source_types == {"INFRA"} and status == "pending":
            origin, basis = "Infrastructure (Pending)", "source_candidates"
        elif len(source_types) == 1:
            origin, basis = labels[next(iter(source_types))], "source_candidates"
        else:
            origin = (
                "Uncertain"
                if source_types
                else ("Pending" if status == "pending" else "Unknown")
            )
            basis = "source_candidates" if source_types else "no_support"
        dark_group = None
        if "DARK" in source_types:
            dark_group = "dark_only" if source_types == {"DARK"} else "dark_mixed"
            if human == "Vessel":
                dark_group = "reviewed_vessel_" + dark_group
            elif human:
                dark_group = "conflicting_human_origin"
        results.append(
            {
                "slick_id": slick["id"],
                "possible_origin": origin,
                "basis": basis,
                "hitl_cls": review,
                "hitl_cls_name": slick["hitl_cls_name"],
                "positive_source_types": sorted(source_types),
                "source_analysis_status": status,
                "dark_group": dark_group,
            }
        )
    return results


area_sources = sources_for_slicks(area_data["features"], "area-sources")
origin_rows = possible_origins(area_data["features"], area_sources)
pd.DataFrame(origin_rows)

## 6. Export complete data and retain provenance

Save the combined complete GeoJSON as a derived file, keeping raw pages unchanged. CSV geometry from the API is EWKT (`SRID=4326;...`). The direct CSV request below is suitable for this bounded example only when all matching rows fit in one page; it verifies the row count before accepting the download. Larger exports require pagination too.

In [ ]:
(OUTPUT_DIR / "area-derived.geojson").write_text(json.dumps(area_data))
(OUTPUT_DIR / "origin-estimates.json").write_text(json.dumps(origin_rows, indent=2))
if area_data["numberMatched"] > 50000:
    raise RuntimeError("This direct CSV example needs pagination for this population")
csv_path = download(
    "public.slick_plus",
    {**AREA_PARAMS, "f": "csv", "limit": 50000},
    "area",
    suffix="csv",
)
with csv_path.open(newline="") as stream:
    csv_count = sum(1 for _ in csv.DictReader(stream))
if csv_count != area_data["numberMatched"]:
    raise RuntimeError(f"CSV is partial or population changed: {csv_count} rows")
print("Complete CSV:", csv_path)

## 7. Retrieve a specific detection and open its evidence link

Fetch a known detection by ID. A missing detection may have been excluded or deactivated; it does not establish an absence of oil. Merge attribution parameters into returned links before opening them, preserving the original in the raw response.

In [ ]:
single_data = fetch_features(
    "public.slick_plus", {"ids": "4111397", "properties": SLICK_PROPERTIES}, "single"
)
for feature in single_data["features"]:
    print(tagged_link(feature["properties"]["slick_url"]))

## 8. Retrieve detections from a Sentinel-1 scene

This retains the same quality screen and acquisition period. A scene can contain multiple detections, which need not be independent events.

In [ ]:
if area_data["features"]:
    scene_id = area_data["features"][0]["properties"]["s1_scene_id"]
    scene_data = fetch_features(
        "public.slick_plus",
        {
            "s1_scene_id": scene_id,
            "datetime": PERIOD,
            "filter": SCREEN,
            "properties": SLICK_PROPERTIES,
        },
        "scene",
    )

## 9. EEZ, IHO sea area or MPA membership

Look up an area's `aoi_id`, query a bounding box that contains its full outline, then filter the returned `aoi_type_N_ids` array **locally**. Do not use the legacy `get_slicks_by_*` functions: they omit the geometric score and have fragile argument/performance behavior. Membership is precomputed intersection, not clipped geometry.

Italy's EEZ uses Marine Regions MRGID 5682. IHO areas use `public.aoi_iho` / `mrgid`; MPAs use `public.aoi_mpa` / `wdpaid`.

In [ ]:
eez_lookup = fetch_features(
    "public.aoi_eez",
    {"mrgid": 5682},
    "italy-lookup",
    key_fields=("aoi_id",),
    sortby="aoi_id",
)
if len(eez_lookup["features"]) != 1:
    raise RuntimeError("Resolve the intended EEZ before querying detections")
aoi_id = eez_lookup["features"][0]["properties"]["aoi_id"]
outline = fetch_features("public.aoi", {"ids": aoi_id}, "italy-outline")
if len(outline["features"]) != 1:
    raise RuntimeError("The selected EEZ outline could not be retrieved")
aoi_bounds = gpd.GeoDataFrame.from_features(outline["features"], crs=4326).total_bounds
aoi_bbox = ",".join(map(str, aoi_bounds))
eez_box_data = fetch_features(
    "public.slick_plus", {**AREA_PARAMS, "bbox": aoi_bbox}, "italy-box"
)
eez_features = [
    feature
    for feature in eez_box_data["features"]
    if aoi_id in (feature["properties"].get("aoi_type_1_ids") or [])
]
print("Italy EEZ potential slick detections:", len(eez_features))
eez_scenes = coverage_context(
    aoi_bbox, PERIOD, "italy-coverage", outline["features"][0]["geometry"]
)

## 10. Query a particular vessel or structure

The same external identifier field can contain a vessel MMSI, a structure ID or a DARK identifier. Specify the source type too. This screen approximates the map's top-three candidate search; source verification/rejections are not exposed, so inspect the source on the map before relying on it. The acquisition period comes from `slick_plus`, **not** association `create_time`.

`git_tag` is provenance and can contain a commit hash. Do not parse it as a guaranteed semantic version or assume a higher tag guarantees better evidence. `create_time` is not a reliable last-modified watermark; updates, deactivation and human review can occur without a new source creation timestamp.

In [ ]:
vessel_matches = fetch_features(
    "public.source_plus",
    {
        "mmsi_or_structure_id": "215371000",
        "source_type": "VESSEL",
        "filter": "source_rank <= 3 AND source_collated_score >= 0",
        "bbox-only": "true",
        "properties": "slick_id,source_id,source_type,source_rank,source_collated_score,create_time,git_tag,source_url",
    },
    "vessel-matches",
    key_fields=("slick_id", "source_id"),
    sortby="slick_id,source_id",
)
vessel_ids = sorted(
    {feature["properties"]["slick_id"] for feature in vessel_matches["features"]}
)
vessel_slicks = []
for start in range(0, len(vessel_ids), 100):
    batch = ",".join(map(str, vessel_ids[start : start + 100]))
    result = fetch_features(
        "public.slick_plus",
        {
            "ids": batch,
            "datetime": PERIOD,
            "filter": SCREEN,
            "properties": SLICK_PROPERTIES,
        },
        f"vessel-slicks-{start}",
    )
    vessel_slicks.extend(result["features"])
print(
    "Potential detections with screened candidate associations in the period:",
    len(vessel_slicks),
)

## 11. Count DARK-only and mixed-type support

For "how many dark vessel slicks", count **unique detections**, separating DARK as the sole positive-scoring type from DARK among several positive-scoring types. Never filter to DARK before building the distinct-type sets. The latter group has uncertain origin; the former has more consistent candidate support, not a calibrated probability or confirmation.

Human-reviewed Vessel detections with DARK support form separate subgroups; conflicting Natural/Infrastructure reviews retain their human origin and are not automatically counted as DARK origins. A human Vessel review alone is not DARK evidence. Several DARK candidates for one slick still count once. For a different area/year, change the population's bbox and UTC acquisition period, then rerun the complete retrieval and grouping.

In [ ]:
dark_groups = {
    name: [row["slick_id"] for row in origin_rows if row["dark_group"] == name]
    for name in (
        "dark_only",
        "dark_mixed",
        "reviewed_vessel_dark_only",
        "reviewed_vessel_dark_mixed",
        "conflicting_human_origin",
    )
}
print(
    json.dumps(
        {
            name: {"count": len(ids), "slick_ids": ids}
            for name, ids in dark_groups.items()
        },
        indent=2,
    )
)

## 12. Latest human review

Use `hitl_cls` and `hitl_cls_name`. Classes 2–8 represent human-classified apparent oil; 9 is Ambiguous. Vessel subclasses 5–8 should be included together when asking for vessel-reviewed detections. Human review interprets the same imagery and is not independent confirmation. Avoid `IS NOT NULL`; use explicit class values.

In [ ]:
classes = fetch_features("public.cls", {}, "classes")
display(pd.DataFrame([feature["properties"] for feature in classes["features"]]))
reviewed_data = fetch_features(
    "public.slick_plus",
    {**AREA_PARAMS, "filter": SCREEN + " AND hitl_cls IN (2,3,4,5,6,7,8)"},
    "human-classified",
)
print("Human-classified apparent-oil detections:", reviewed_data["numberMatched"])

## Reporting and attribution

Report the scope (UTC dates, area, intersects/not clipped), filters and deliberate changes, unique detections retrieved versus `numberMatched`, retrieval time, processing/coverage limits, and human versus automated evidence. Polygon area is not oil volume, and overlapping areas cannot be summed as unique pollution. Zero matches do not establish a clean area. Source associations do not establish responsibility or legality.

Credit **SkyTruth Cerulean**, provide the [CC BY-SA 4.0 license](https://creativecommons.org/licenses/by-sa/4.0/) and disclose filtering/analysis modifications. Consult the [terms](https://skytruth.org/terms-of-service) and the [interpretation guide](https://api.cerulean.skytruth.org/guide/methodology?utm_medium=notebook&utm_source=unidentified-colab). Do not imply SkyTruth reviewed your analysis. Keep credentials, reviewer identities and sensitive community locations out of shared outputs.

Inspect consequential detections using the tagged map links, imagery, wind and candidate tracks. Seek independent evidence before stronger conclusions. All downloaded files and the reproducible request log are in `cerulean-api-results/`.